# 🎮 MineColab Modded

Servidor de **Minecraft Java con mods** en Google Colab, pensado para sesiones
cortas (~2 h), no para tenerlo 24/7.

Basado en la idea de [thecoder-001/MineColab](https://github.com/thecoder-001/MineColab),
reescrito para versiones actuales (1.21.x, 26.x…), con **selector de versión y
loader**, descarga de **mods desde Modrinth** con sus dependencias, y **túnel**
(playit.gg con dirección fija, o bore.pub sin crear cuenta) para que entren tus
amigos sin abrir puertos.

### Cómo funciona

```
Google Drive (persistente)              Colab (rápido, se borra al terminar)
MineColab/server/  ──copia al inicio──▶  /content/mcserver/  ──▶  java … server.jar
   ├─ mods/                                (el servidor corre aquí)      │
   ├─ world/                                                             ▼
   └─ config/                                                    playit.gg / ngrok
MineColab/backups/  ◀── guardado al apagar / cada 10 min
```

El mundo, los mods y la configuración viven en **tu Drive**, así que puedes
cerrar la sesión y volver otro día sin perder nada.

### Orden de ejecución (una vez por sesión)

| # | Celda | Qué hace |
|---|-------|----------|
| 1 | Configuración | versión, loader, RAM, horas |
| 1b | Mods y ajustes | lista de mods, túnel, whitelist |
| 2 | Preparar entorno | monta Drive, instala Java, copia el servidor |
| 3 | Instalar servidor | descarga el jar/instalador del loader elegido |
| 4 | Mods | resuelve y descarga mods de Modrinth |
| 5 | Ajustes | `server.properties`, whitelist, admins |
| 6 | **Arrancar** | túnel + servidor + autoguardado + apagado a las 2 h |
| 7 | Buscar mods | buscador de Modrinth (opcional) |
| 8 | Extras | guardar ya, backups, restaurar, ver logs |

> ⚠️ **Antes de empezar**
> - Ejecuta las celdas **en orden**. La celda 6 se queda corriendo: mientras
>   esté activa, el servidor vive.
> - Mantén la pestaña de Colab abierta. Si Colab recicla la máquina, el servidor
>   muere (por eso el autoguardado cada 10 min).
> - Colab es gratis para uso interactivo; un servidor de juego consume recursos
>   que no están pensados para esto. Úsalo de forma moderada y no como hosting
>   permanente. Usar varias cuentas para saltarse límites puede hacer que te
>   restrinjan el acceso.
> - RAM del plan gratuito: ~12 GB en total. Deja 6–8 GB para el servidor.
> - Los **mods deben coincidir** en versión y loader entre el servidor y cada
>   cliente (tus amigos). Los mods de cliente (shaders, minimapa) van solo en el
>   cliente; este notebook omite automáticamente los que son `client_only`.
> - ¿Un mod o plugin no está en Modrinth? Sube el `.jar` a `MineColab/server/mods`
>   (o `.../plugins` en paper) desde Google Drive y aparecerá en la próxima sesión.
>   El notebook solo borra los archivos que él mismo descargó.
> - 🔐 **Seguridad:** montar Drive da a la VM acceso a **toda** tu unidad, y los
>   mods son código de terceros que corre como root. Usa una cuenta de Google
>   dedicada para el servidor, deja `ONLINE_MODE = true` y pon una `WHITELIST`.
>   Todo lo que se descarga se verifica por hash antes de ejecutarse (ver README).


In [ ]:
#@title ⚙️ 1. Configuración del servidor { display-mode: "form" }
#@markdown Elige loader, versión y túnel. Luego baja a la celda siguiente para los mods.

SERVER_TYPE = "fabric" #@param ["fabric", "neoforge", "forge", "paper", "vanilla"]
MC_VERSION = "1.21.1" #@param {type:"string"}
TUNNEL = "playit" #@param ["playit", "bore", "ngrok", "ninguno"]
RAM_GB = 6 #@param [2, 3, 4, 5, 6, 8, 10, 12] {type:"raw"}
MAX_HORAS = 2 #@param {type:"number"}

# ── Notas rápidas ────────────────────────────────────────────────────────────
# fabric    → el mejor para mods. Necesita Fabric API (se añade solo).
# neoforge  → modpacks modernos (1.20.2 en adelante).
# forge     → modpacks clásicos (1.12.2, 1.16.5, 1.20.1…).
# paper     → sin mods, pero plugins y muy optimizado.
# vanilla   → el servidor oficial, sin mods ni plugins.
#
# MC_VERSION debe existir en Mojang y tener build del loader:
#   1.21.1, 1.21.4, 1.21.8, 26.1, 26.3 …
# Si un mod no tiene versión para la tuya, baja a una anterior (1.21.1 es la
# versión con más mods).
#
# TUNNEL: playit.gg = dirección fija y gratis (recomendado, requiere cuenta).
#         bore      = cero configuración, dirección temporal (bore.pub:12345).
#         ngrok     = su plan gratis ya no incluye TCP.
#         ninguno   = solo dentro de la sesión.

print(f"🎮 {SERVER_TYPE} · Minecraft {MC_VERSION} · {RAM_GB} GB de RAM")
print(f"🔗 túnel: {TUNNEL} · ⏱️  apagado automático a las {MAX_HORAS} h")


In [ ]:
#@title 🧩 1b. Mods y ajustes finos
#@markdown Escribe los **slugs de Modrinth** (la celda de búsqueda del final te ayuda a encontrarlos).
#@markdown También puedes pegar URLs directas a un `.jar` (CurseForge, GitHub, etc.).

# ── Mods del servidor (slugs de Modrinth o URLs .jar) ───────────────────────
MODS = [
    # "fabric-api",      # se añade automáticamente en fabric
    # "lithium",         # optimización de ticks (servidor + cliente)
    # "ferrite-core",    # menos uso de RAM
    # "chunky",          # pre-generar chunks sin lag
    # "spark",           # profiler: /spark tps
    # "https://ejemplo.com/mi-mod.jar",
]

# En paper/vanilla esto se descarga en plugins/ o se ignora.
PLUGINS = [
    # "luckperms",
    # "essentialsx",
]

# ── Red y jugadores ────────────────────────────────────────────────────────
MOTD = "MineColab — sesión de 2h"
MAX_PLAYERS = 10
VIEW_DISTANCE = 8            # 6–10 va bien en Colab; más = más lag
ONLINE_MODE = True           # False solo si usas clientes no oficiales
WHITELIST = []               # ["TuNombre", "Amigo123"]
OPS = []                     # admins: ["TuNombre"]
BACKUP_MIN = 10              # autoguardado a Drive cada N minutos
SEED = ""                    # "" = aleatoria

# ── Túnel (solo si TUNNEL == "ngrok") ─────────────────────────────────────
NGROK_AUTHTOKEN = ""         # https://dashboard.ngrok.com/get-started/your-authtoken

EXTRA_JVM_ARGS = ""          # p. ej. "-Dfml.queryResult=confirm"

print(f"Mods configurados: {len(MODS)} · plugins: {len(PLUGINS)}")


In [ ]:
#@title 🧰 2. Preparar entorno (Drive + Java) { display-mode: "form" }
#@markdown Monta Drive, crea las carpetas, detecta la Java que pide tu versión y trae el servidor. ~1 min.

import os, sys, re, json, glob, time, shutil, socket, hashlib, tarfile, subprocess, threading
import urllib.request, urllib.parse
from datetime import datetime
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/MineColab")
    SERVER_DIR = Path("/content/mcserver")
else:
    DRIVE_ROOT = Path(os.environ.get("MCCOLAB_DRIVE", str(Path.home() / "MineColab")))
    SERVER_DIR = Path(os.environ.get("MCCOLAB_SERVER", "/private/tmp/opencode/mcserver"))
    print("ℹ️  Fuera de Colab: usando carpetas locales (modo prueba).")

DRIVE_SERVER = DRIVE_ROOT / "server"
BACKUP_DIR = DRIVE_ROOT / "backups"
for _p in (DRIVE_SERVER, BACKUP_DIR, SERVER_DIR):
    _p.mkdir(parents=True, exist_ok=True)

UA = {"User-Agent": "MineColab-Modded/2.0 (Google Colab notebook)"}


def get_json(url, timeout=45):
    req = urllib.request.Request(url, headers=UA)
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode("utf-8"))


def download(url, dest, sha512=None, sha256=None, sha1=None, label=None):
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    req = urllib.request.Request(url, headers=UA)
    with urllib.request.urlopen(req, timeout=600) as r, open(dest, "wb") as f:
        shutil.copyfileobj(r, f)
    expected = sha512 or sha256 or sha1
    if expected:
        algo = hashlib.sha512 if sha512 else (hashlib.sha256 if sha256 else hashlib.sha1)
        got = algo(dest.read_bytes()).hexdigest()
        if got.lower() != expected.lower():
            dest.unlink(missing_ok=True)
            raise RuntimeError(f"Hash incorrecto en {dest.name}: esperaba {expected[:16]}… y llegó {got[:16]}…")
        print(f"   🔒 hash {algo().name} verificado")
    print(f"   ↓ {label or dest.name} ({dest.stat().st_size / 1e6:.1f} MB)")
    return dest


def fetch_text(url, timeout=30):
    req = urllib.request.Request(url, headers=UA)
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read().decode("utf-8", "replace").strip().split()[0]


def github_sha256(repo, tag, asset_name):
    """GitHub publica el sha256 de cada asset de release; si falla, seguimos sin hash."""
    try:
        suffix = "latest" if tag == "latest" else f"tags/{tag}"
        release = get_json(f"https://api.github.com/repos/{repo}/releases/{suffix}")
        for asset in release.get("assets", []):
            if asset.get("name") == asset_name:
                digest = (asset.get("digest") or "").replace("sha256:", "")
                return digest or None
    except Exception as exc:
        print(f"   (no pude consultar el hash de {asset_name}: {exc})")
    return None


def safe_extract(archive, dest):
    """Extrae un tar.gz comprobando que ninguna entrada salga del destino."""
    dest = Path(dest).resolve()
    with tarfile.open(archive) as tar:
        for member in tar.getmembers():
            target = (dest / member.name).resolve()
            if target != dest and not str(target).startswith(str(dest) + os.sep):
                raise RuntimeError(f"entrada sospechosa en {Path(archive).name}: {member.name}")
            if member.issym() or member.islnk():
                raise RuntimeError(f"enlace sospechoso en {Path(archive).name}: {member.name}")
        try:
            tar.extractall(dest, filter="data")
        except TypeError:
            tar.extractall(dest)


def sh(cmd, check=False, quiet=True):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.returncode != 0:
        if check:
            print(r.stdout[-3000:])
            print(r.stderr[-3000:])
            raise RuntimeError(f"Falló el comando: {cmd}")
        if not quiet:
            print(r.stdout[-2000:], r.stderr[-2000:])
    return r


# ── Metadatos oficiales de Mojang (versión + Java requerida) ────────────────
_MANIFEST = None


def mc_manifest():
    global _MANIFEST
    if _MANIFEST is None:
        _MANIFEST = get_json("https://piston-meta.mojang.com/mc/game/version_manifest_v2.json")
    return _MANIFEST


def mc_meta(version):
    for v in mc_manifest()["versions"]:
        if v["id"] == version:
            return get_json(v["url"])
    raise RuntimeError(f"Minecraft {version} no existe. Revisa MC_VERSION (ej: 1.21.1, 26.3).")


def _java_major(exe):
    try:
        out = subprocess.run([exe, "-version"], capture_output=True, text=True).stderr
        m = re.search(r'version "(\d+)(?:\.(\d+))?', out)
        if not m:
            return None
        major = int(m.group(1))
        return 8 if major == 1 else major
    except Exception:
        return None


def ensure_java(major):
    major = int(major)
    if sys.platform != "linux":
        exe = shutil.which("java")
        print("ℹ️  Sistema no Linux: uso la Java del sistema.", exe or "")
        return exe or "java"
    candidates = [f"/usr/lib/jvm/java-{major}-openjdk-amd64/bin/java"]
    if major == 8:
        candidates.append("/usr/lib/jvm/java-8-openjdk-amd64/bin/java")
    for c in candidates:
        if os.path.exists(c) and _java_major(c) == major:
            return c
    current = shutil.which("java")
    if current and _java_major(current) == major:
        return current
    if shutil.which("apt-get"):
        print(f"   Instalando Java {major} con apt…")
        sh("sudo apt-get update -qq")
        pkg = "openjdk-8-jre-headless" if major == 8 else f"openjdk-{major}-jre-headless"
        sh(f"sudo DEBIAN_FRONTEND=noninteractive apt-get install -y -qq {pkg}")
        for c in candidates:
            if os.path.exists(c) and _java_major(c) == major:
                return c
    print(f"   Descargando Java {major} (Eclipse Temurin)…")
    assets = get_json(
        f"https://api.adoptium.net/v3/assets/latest/{major}/hotspot"
        "?architecture=x64&image_type=jre&os=linux&vendor=eclipse"
    )
    package = assets[0]["binary"]["package"]
    tgz = Path(f"/tmp/temurin-{major}.tar.gz")
    download(package["link"], tgz, sha256=package["checksum"])
    out = Path(f"/opt/temurin-{major}")
    shutil.rmtree(out, ignore_errors=True)
    out.mkdir(parents=True, exist_ok=True)
    safe_extract(tgz, out)
    exe = glob.glob(f"{out}/*/bin/java")[0]
    os.chmod(exe, 0o755)
    return exe


# ── Sincronización Drive ⇄ disco local ─────────────────────────────────────
HAS_RSYNC = shutil.which("rsync") is not None
if not HAS_RSYNC and shutil.which("apt-get"):
    sh("sudo apt-get update -qq && sudo apt-get install -y -qq rsync")
    HAS_RSYNC = shutil.which("rsync") is not None
if not HAS_RSYNC:
    print("ℹ️  rsync no está disponible: usaré copia directa (más lenta).")

RSYNC_OPTS = "--no-perms --no-owner --no-group --no-times --exclude logs --exclude cache --exclude crash-reports"


def sync_from_drive(clean=False):
    if clean:
        shutil.rmtree(SERVER_DIR, ignore_errors=True)
        SERVER_DIR.mkdir(parents=True, exist_ok=True)
    if not HAS_RSYNC:
        shutil.copytree(DRIVE_SERVER, SERVER_DIR, dirs_exist_ok=True)
        return
    r = sh(f'rsync -r {RSYNC_OPTS} "{DRIVE_SERVER}/" "{SERVER_DIR}/"')
    if r.returncode not in (0, 23, 24):
        print("⚠️  rsync avisó de un problema al copiar desde Drive.")


def sync_to_drive():
    print("📤 Guardando en Drive…")
    if not HAS_RSYNC:
        shutil.copytree(SERVER_DIR, DRIVE_SERVER, dirs_exist_ok=True)
        print("   ✅ guardado")
        return
    r = sh(f'rsync -r --delete {RSYNC_OPTS} --exclude "*.log" "{SERVER_DIR}/" "{DRIVE_SERVER}/"')
    print("   ✅ guardado" if r.returncode in (0, 23, 24) else "   ⚠️  revisa el aviso de rsync")


def sync_worlds():
    for w in sorted(SERVER_DIR.glob("world*")):
        if not w.is_dir():
            continue
        if HAS_RSYNC:
            sh(f'rsync -r --delete --no-perms --no-owner --no-group --no-times "{w}" "{DRIVE_SERVER}/"')
        else:
            shutil.copytree(w, DRIVE_SERVER / w.name, dirs_exist_ok=True)


def backup_world(tag="manual"):
    worlds = [w for w in SERVER_DIR.glob("world*") if w.is_dir()]
    if not worlds:
        return None
    out = BACKUP_DIR / f"world-{tag}-{datetime.now():%Y%m%d-%H%M%S}.tar.gz"
    with tarfile.open(out, "w:gz") as t:
        for w in worlds:
            t.add(w, arcname=w.name)
    for old in sorted(BACKUP_DIR.glob("world-*.tar.gz"))[:-3]:
        old.unlink(missing_ok=True)
    print(f"💾 backup: {out.name} ({out.stat().st_size / 1e6:.0f} MB)")
    return out


# ── Puesta a punto ─────────────────────────────────────────────────────────
def assert_stopped():
    """Evita tocar archivos mientras el servidor está escribiendo el mundo."""
    proc = globals().get("SERVER_PROC")
    if proc is not None and proc.poll() is None:
        raise RuntimeError("Hay un servidor corriendo. Para la celda 6 antes de continuar.")


assert_stopped()
if not re.fullmatch(r"[0-9A-Za-z][0-9A-Za-z._+-]{0,31}", str(MC_VERSION)):
    raise RuntimeError(f"MC_VERSION='{MC_VERSION}' tiene caracteres no válidos.")
META = mc_meta(MC_VERSION)
JAVA_MAJOR = int(META["javaVersion"]["majorVersion"])
print(f"🎮 Minecraft {MC_VERSION} pide Java {JAVA_MAJOR}")
JAVA = os.environ.get("MCCOLAB_JAVA") or ensure_java(JAVA_MAJOR)
os.environ["JAVA_HOME"] = str(Path(JAVA).resolve().parent.parent)
os.environ["PATH"] = str(Path(JAVA).resolve().parent) + os.pathsep + os.environ.get("PATH", "")
print(f"☕ Java: {JAVA}")

first_time = not (SERVER_DIR / "server.properties").exists()
sync_from_drive(clean=first_time)
print(f"📁 Servidor local: {SERVER_DIR}")
print(f"☁️  Persistencia:   {DRIVE_ROOT}")
print("\n➡️  Siguiente: ejecuta la celda 3 para instalar el servidor.")


In [ ]:
#@title 📦 3. Instalar el servidor { display-mode: "form" }
#@markdown Descarga el loader elegido. Es idempotente: si ya está instalado, no vuelve a bajar nada.

import json, re, shlex, shutil, glob
from pathlib import Path

STATE_FILE = SERVER_DIR / ".minecolab_state.json"
EULA_FILE = SERVER_DIR / "eula.txt"


def ram_disponible_gb():
    """RAM real de la VM (cgroup de Colab), no la del host."""
    candidatos = []
    for ruta in ("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory/memory.limit_in_bytes"):
        try:
            valor = Path(ruta).read_text().strip()
            if valor.isdigit():
                candidatos.append(int(valor) / 1e9)
        except Exception:
            pass
    try:
        for linea in Path("/proc/meminfo").read_text().splitlines():
            if linea.startswith("MemTotal:"):
                candidatos.append(int(linea.split()[1]) / 1e6)
                break
    except Exception:
        pass
    candidatos = [c for c in candidatos if 1 < c < 100000]
    return min(candidatos) if candidatos else None


def jvm_flags():
    global RAM_EFECTIVA_GB
    gb = int(RAM_GB)
    limite = ram_disponible_gb()
    if limite:
        seguro = max(2, int(limite - 2))
        if gb > seguro:
            print(f"⚠️  RAM_GB={gb} no cabe en esta VM ({limite:.1f} GB). Bajo a {seguro} GB.")
            gb = seguro
    RAM_EFECTIVA_GB = gb
    flags = [
        f"-Xms{min(gb, 2)}G", f"-Xmx{gb}G",
        "-XX:+UseG1GC", "-XX:+ParallelRefProcEnabled", "-XX:MaxGCPauseMillis=200",
        "-XX:+UnlockExperimentalVMOptions", "-XX:+DisableExplicitGC",
        "-XX:+ExitOnOutOfMemoryError",
        "-XX:G1NewSizePercent=30", "-XX:G1MaxNewSizePercent=40", "-XX:G1HeapRegionSize=8M",
        "-XX:G1ReservePercent=20", "-XX:G1HeapWastePercent=5", "-XX:G1MixedGCCountTarget=4",
        "-XX:InitiatingHeapOccupancyPercent=15", "-XX:G1MixedGCLiveThresholdPercent=90",
        "-XX:G1RSetUpdatingPauseTimePercent=5", "-XX:SurvivorRatio=32",
        "-XX:+PerfDisableSharedMem", "-XX:MaxTenuringThreshold=1",
    ]
    if EXTRA_JVM_ARGS.strip():
        flags += EXTRA_JVM_ARGS.split()
    return flags


def vkey(v):
    return tuple(int(x) for x in re.findall(r"\d+", v))


# ── Resolución de descargas (url, detalle, hashes) ──────────────────────────
def resolve_vanilla(mc):
    meta = mc_meta(mc)
    server = meta["downloads"]["server"]
    return server["url"], "oficial", {"sha1": server["sha1"]}


def resolve_paper(mc):
    data = get_json(f"https://fill.papermc.io/v3/projects/paper/versions/{mc}/builds")
    if not isinstance(data, list) or not data:
        raise RuntimeError(f"Paper no tiene builds para {mc} (prueba 1.21.8, 1.21.1, 26.3…).")
    builds = [b for b in data if b.get("channel") == "STABLE"] or data
    b = builds[0]
    jar = b["downloads"]["server:default"]
    return jar["url"], f"build {b['id']}", {"sha256": jar["checksums"]["sha256"]}


def resolve_fabric(mc):
    inst = next(i for i in get_json("https://meta.fabricmc.net/v2/versions/installer") if i.get("stable"))
    loaders = get_json(f"https://meta.fabricmc.net/v2/versions/loader/{urllib.parse.quote(mc)}")
    if not loaders:
        raise RuntimeError(f"Fabric no soporta {mc} todavía.")
    return inst["url"], loaders[0]["loader"]["version"], {"sha256": fetch_text(inst["url"] + ".sha256")}


def resolve_neoforge(mc):
    prefix = "47." if mc == "1.20.1" else (mc[2:] if mc.startswith("1.") else mc) + "."
    versions = get_json(
        "https://maven.neoforged.net/api/maven/versions/releases/net/neoforged/neoforge"
    )["versions"]
    cands = sorted({v for v in versions if v.startswith(prefix) and "-" not in v}, key=vkey)
    if not cands:
        raise RuntimeError(f"NeoForge no tiene versión para {mc} (prueba 1.21.1, 1.21.8, 26.3…).")
    v = cands[-1]
    url = f"https://maven.neoforged.net/releases/net/neoforged/neoforge/{v}/neoforge-{v}-installer.jar"
    return url, v, {"sha1": fetch_text(url + ".sha1")}


def resolve_forge(mc):
    promos = get_json("https://files.minecraftforge.net/net/minecraftforge/forge/promotions_slim.json")["promos"]
    fv = promos.get(f"{mc}-recommended") or promos.get(f"{mc}-latest")
    if not fv:
        raise RuntimeError(
            f"Forge no publica para {mc}. Usa neoforge/fabric o una versión clásica (1.12.2, 1.16.5, 1.20.1)."
        )
    url = f"https://maven.minecraftforge.net/net/minecraftforge/forge/{mc}-{fv}/forge-{mc}-{fv}-installer.jar"
    return url, fv, {"sha1": fetch_text(url + ".sha1")}


def resolve_any(kind, mc):
    fn = {"vanilla": resolve_vanilla, "paper": resolve_paper, "fabric": resolve_fabric,
          "neoforge": resolve_neoforge, "forge": resolve_forge}[kind]
    return fn(mc)


# ── Comando de arranque ────────────────────────────────────────────────────
def launch_argv():
    if SERVER_TYPE in ("vanilla", "paper"):
        return [JAVA, *jvm_flags(), "-jar", "server.jar", "nogui"]
    if SERVER_TYPE == "fabric":
        jar = "fabric-server-launch.jar"
        if not (SERVER_DIR / jar).exists():
            found = sorted(SERVER_DIR.glob("fabric-server-*.jar"))
            if not found:
                raise RuntimeError("No encuentro el lanzador de Fabric. Reinstala la celda 3.")
            jar = found[0].name
        return [JAVA, *jvm_flags(), "-jar", jar, "nogui"]
    if (SERVER_DIR / "run.sh").exists():
        for pattern in (
            "libraries/net/neoforged/neoforge/*/unix_args.txt",
            "libraries/net/minecraftforge/forge/*/unix_args.txt",
        ):
            found = sorted(SERVER_DIR.glob(pattern))
            if found:
                args_file = found[0].relative_to(SERVER_DIR)
                return [JAVA, "@user_jvm_args.txt", "@" + str(args_file), "nogui"]
        return ["bash", "run.sh", "nogui"]
    for pattern in ("forge-*-universal.jar", "neoforge-*-universal.jar"):
        found = sorted(SERVER_DIR.glob(pattern))
        if found:
            return [JAVA, *jvm_flags(), "-jar", found[0].name, "nogui"]
    raise RuntimeError("No encuentro cómo arrancar Forge/NeoForge. Reinstala la celda 3.")


def load_state():
    try:
        return json.loads(STATE_FILE.read_text())
    except Exception:
        return {}


def save_state(d):
    STATE_FILE.write_text(json.dumps(d, indent=2))


KEEP = {
    "world", "world_nether", "world_the_end", "mods", "plugins", "config", "defaultconfigs",
    "server.properties", "eula.txt", "ops.json", "whitelist.json", "banned-ips.json",
    "banned-players.json", "usercache.json", "server-icon.png", "user_jvm_args.txt",
    ".minecolab_state.json", ".minecolab_mods.json",
}


def clean_old_install():
    for entry in SERVER_DIR.iterdir():
        if entry.name in KEEP:
            continue
        if entry.is_dir():
            shutil.rmtree(entry, ignore_errors=True)
        else:
            entry.unlink(missing_ok=True)


# === INSTALL ===
assert_stopped()
state = load_state()
already = (
    state.get("type") == SERVER_TYPE
    and state.get("mc") == MC_VERSION
    and EULA_FILE.exists()
)
if already:
    try:
        launch_argv()
    except Exception:
        already = False

if already:
    print(f"✅ Ya instalado: {SERVER_TYPE} {MC_VERSION} (nada que descargar)")
else:
    print(f"📦 Instalando {SERVER_TYPE} {MC_VERSION}…")
    clean_old_install()
    url, extra, hashes = resolve_any(SERVER_TYPE, MC_VERSION)
    if SERVER_TYPE in ("fabric", "forge", "neoforge") and not re.fullmatch(
        r"[0-9A-Za-z._+-]{1,64}", str(extra)
    ):
        raise RuntimeError(f"Versión de loader no válida: {extra!r}")
    print(f"   fuente: {url.split('/')[2]} · {extra}")

    if SERVER_TYPE in ("vanilla", "paper"):
        download(url, SERVER_DIR / "server.jar", **hashes)
    elif SERVER_TYPE == "fabric":
        installer = download(url, SERVER_DIR / "fabric-installer.jar", **hashes)
        sh(
            f"cd {shlex.quote(str(SERVER_DIR))} && {shlex.quote(JAVA)} -jar "
            f"{shlex.quote(installer.name)} server -mcversion {shlex.quote(str(MC_VERSION))} "
            f"-loader {shlex.quote(str(extra))} -downloadMinecraft",
            check=True, quiet=False,
        )
    else:
        installer = download(url, SERVER_DIR / "installer.jar", **hashes)
        sh(
            f"cd {shlex.quote(str(SERVER_DIR))} && {shlex.quote(JAVA)} -jar "
            f"{shlex.quote(installer.name)} --installServer",
            check=True, quiet=False,
        )
        installer.unlink(missing_ok=True)
        (SERVER_DIR / "installer.jar.log").unlink(missing_ok=True)

    if SERVER_TYPE in ("forge", "neoforge"):
        # run.sh / unix_args leen los flags de aquí; se reescribe cada vez por si cambias RAM_GB
        (SERVER_DIR / "user_jvm_args.txt").write_text("\n".join(jvm_flags()) + "\n")

    EULA_FILE.write_text(
        "# Al poner eula=true aceptas la EULA de Minecraft: https://aka.ms/MinecraftEULA\n"
        "eula=true\n"
    )
    save_state({"type": SERVER_TYPE, "mc": MC_VERSION, "loader": extra})
    print("✅ Instalación terminada")

if SERVER_TYPE in ("forge", "neoforge"):
    (SERVER_DIR / "user_jvm_args.txt").write_text("\n".join(jvm_flags()) + "\n")

missing = [p for p in ("eula.txt",) if not (SERVER_DIR / p).exists()]
if missing:
    print("⚠️  Falta:", missing)
print("\n➡️  Siguiente: celda 4 (mods) y celda 5 (ajustes).")


In [ ]:
#@title 🧩 4. Descargar mods desde Modrinth { display-mode: "form" }
#@markdown Resuelve la versión compatible con tu `MC_VERSION` + loader, baja dependencias y limpia las que ya no uses.

import json, hashlib, re, urllib.parse
from pathlib import Path

MODRINTH = "https://api.modrinth.com/v2"
LOADER_TAGS = {
    "fabric": ["fabric"],
    "forge": ["forge"],
    "neoforge": ["neoforge"],
    "paper": ["paper", "bukkit", "spigot"],
    "vanilla": [],
}
LOADERS = LOADER_TAGS.get(SERVER_TYPE, [])
IS_PLUGIN_SERVER = SERVER_TYPE == "paper"
MOD_DIR = SERVER_DIR / ("plugins" if IS_PLUGIN_SERVER else "mods")
MOD_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_FILE = SERVER_DIR / ".minecolab_mods.json"


def modrinth_pick(project_id):
    query = urllib.parse.urlencode({
        "game_versions": json.dumps([MC_VERSION]),
        "loaders": json.dumps(LOADERS),
    })
    versions = get_json(f"{MODRINTH}/project/{project_id}/version?{query}")
    if not versions:
        return None
    for kind in ("release", "beta", "alpha"):
        pool = [v for v in versions if v.get("version_type") == kind]
        if pool:
            return sorted(pool, key=lambda v: v["date_published"])[-1]
    return None


def primary_file(ver):
    files = ver.get("files") or []
    preferred = [f for f in files if f.get("primary")] or files
    return preferred[0]


def resolve(key):
    """Devuelve (proyecto, versión) o lanza RuntimeError con el motivo."""
    key = str(key).strip()
    if not re.fullmatch(r"[A-Za-z0-9_-]{1,64}", key):
        raise RuntimeError("nombre de proyecto no válido (usa el slug de Modrinth)")
    project = get_json(f"{MODRINTH}/project/{urllib.parse.quote(key, safe='')}")
    if project.get("project_type") == "modpack":
        raise RuntimeError("es un modpack, no un mod suelto")
    if project.get("server_side") == "unsupported":
        raise RuntimeError("solo funciona en el cliente")
    ver = modrinth_pick(project["id"])
    if not ver:
        raise RuntimeError(f"sin versión para {MC_VERSION}/{SERVER_TYPE}")
    if ver.get("environment") == "client_only":
        raise RuntimeError("solo funciona en el cliente")
    return project, ver


MODS = globals().get("MODS", [])
PLUGINS = globals().get("PLUGINS", [])

wanted = [str(m).strip() for m in MODS if str(m).strip()]
if IS_PLUGIN_SERVER:
    wanted += [str(p).strip() for p in PLUGINS if str(p).strip()]
if SERVER_TYPE == "fabric" and not any(m.lower() in ("fabric-api", "fabric_api") for m in wanted):
    wanted.insert(0, "fabric-api")
    print("ℹ️  Añadido fabric-api (obligatorio en Fabric).")

direct_urls = [m for m in wanted if m.startswith("http")]
wanted = [m for m in wanted if not m.startswith("http")]

if SERVER_TYPE == "vanilla" and (wanted or direct_urls):
    print("⚠️  vanilla no admite mods ni plugins: se ignora la lista. Usa fabric/neoforge/forge/paper.")
    wanted, direct_urls = [], []

chosen, skipped = {}, []


def add_with_deps(key, depth=0):
    if depth > 4:
        return
    try:
        project, ver = resolve(key)
    except Exception as exc:
        skipped.append((key, str(exc)))
        return
    if project["id"] in chosen:
        return
    chosen[project["id"]] = (project, ver)
    for dep in ver.get("dependencies", []):
        if dep.get("dependency_type") == "required" and dep.get("project_id"):
            add_with_deps(dep["project_id"], depth + 1)


for key in wanted:
    add_with_deps(key)

managed = set()
print(f"\n📦 {len(chosen)} proyecto(s) resueltos:\n")
for project, ver in chosen.values():
    f = primary_file(ver)
    dest = MOD_DIR / f["filename"]
    expected = (f.get("hashes") or {}).get("sha512")
    if dest.exists() and expected and hashlib.sha512(dest.read_bytes()).hexdigest() == expected:
        print(f"   = {project['slug']:<24} {ver['version_number']}")
    else:
        print(f"   + {project['slug']:<24} {ver['version_number']}")
        try:
            download(f["url"], dest, sha512=expected)
        except Exception as exc:
            skipped.append((project["slug"], str(exc)))
            continue
    managed.add(f["filename"])

for url in direct_urls:
    parsed = urllib.parse.urlparse(url)
    if parsed.scheme not in ("http", "https"):
        skipped.append((url, "solo se permiten enlaces http/https"))
        continue
    # El nombre debe ser un .jar simple: nada de "..", barras ni rutas codificadas.
    decoded = urllib.parse.unquote(parsed.path).replace("\\", "/")
    name = decoded.rsplit("/", 1)[-1]
    target = (MOD_DIR / name).resolve()
    if (
        ".." in decoded
        or not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9 ._+()\[\]-]{0,120}\.jar", name, re.IGNORECASE)
        or target.parent != MOD_DIR.resolve()
    ):
        skipped.append((url, "el nombre del archivo no es un .jar válido"))
        continue
    if not target.exists():
        print(f"   + (url) {name}  (sin hash: confías en ese enlace)")
        try:
            download(url, target)
        except Exception as exc:
            target.unlink(missing_ok=True)
            skipped.append((url, f"no se pudo descargar ({exc})"))
            continue
    managed.add(name)

# Quitar mods que gestionamos antes y ya no están en la lista
old_managed = set()
if MANIFEST_FILE.exists():
    try:
        old_managed = set(json.loads(MANIFEST_FILE.read_text()))
    except Exception:
        pass
for name in sorted(old_managed - managed):
    target = MOD_DIR / name
    if target.exists():
        target.unlink()
        print(f"   🗑️  quitado {name}")
MANIFEST_FILE.write_text(json.dumps(sorted(managed), indent=2))

if skipped:
    print("\n⚠️  Omitidos:")
    for name, why in skipped:
        print(f"   · {name}: {why}")
if not chosen and not direct_urls:
    print("\n(No hay mods configurados. Edita la celda 1b cuando quieras.)")

print(f"\n📂 {MOD_DIR} → {len(managed)} archivo(s)")
print("Recuerda: los mods de cliente (shaders, minimapa) van solo en cada cliente.")


In [ ]:
#@title 🛠️ 5. Ajustes del servidor { display-mode: "form" }
#@markdown Escribe `server.properties` conservando lo que ya tengas, y aplica whitelist/ops al arrancar.

from pathlib import Path
import re

PROPS = SERVER_DIR / "server.properties"


def clean_name_list(names):
    """Solo nombres de Minecraft válidos (evita inyectar comandos en la consola)."""
    valid, bad = [], []
    for name in names:
        text = str(name).strip()
        (valid if re.fullmatch(r"[A-Za-z0-9_]{1,16}", text) else bad).append(text)
    if bad:
        print(f"⚠️  Ignorados (nombre no válido): {bad}")
    return valid


WHITELIST = clean_name_list(globals().get("WHITELIST", []))
OPS = clean_name_list(globals().get("OPS", []))
MOTD = re.sub(r"[\r\n]+", " ", str(MOTD))[:120]
SEED = re.sub(r"[\r\n]+", "", str(SEED))[:64]

FORCE = {
    "server-port": "25565",
    "motd": MOTD,
    "max-players": str(MAX_PLAYERS),
    "view-distance": str(VIEW_DISTANCE),
    "simulation-distance": str(max(4, VIEW_DISTANCE - 2)),
    "online-mode": "true" if ONLINE_MODE else "false",
    "enforce-secure-profile": "true" if ONLINE_MODE else "false",
    "allow-flight": "true",
    "enable-command-block": "true",
    "max-tick-time": "-1",
    "spawn-protection": "0",
    "sync-chunk-writes": "false",
    "enable-status": "true",
    "prevent-proxy-connections": "false",
}
if WHITELIST:
    FORCE["white-list"] = "true"
    FORCE["enforce-whitelist"] = "true"
if SEED:
    FORCE["level-seed"] = str(SEED)


def read_props(path):
    data = {}
    if path.exists():
        for line in path.read_text(errors="ignore").splitlines():
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            data[key.strip()] = value.strip()
    return data


props = read_props(PROPS)
created = not props
props.update(FORCE)
PROPS.write_text("#Minecraft server properties\n" + "\n".join(f"{k}={v}" for k, v in props.items()) + "\n")

print(("🆕 Creado" if created else "✏️  Actualizado") + f" {PROPS.name}")
for key in ("server-port", "motd", "max-players", "view-distance", "online-mode", "white-list", "level-seed"):
    if key in props:
        print(f"   {key:<18} = {props[key]}")

if WHITELIST:
    print(f"\n🔐 whitelist: {', '.join(WHITELIST)} (se aplica al arrancar)")
if OPS:
    print(f"👑 admins: {', '.join(OPS)} (se aplican al arrancar)")
print("\n➡️  Siguiente: celda 6, la que arranca el servidor.")


In [ ]:
#@title ▶️ 6. Arrancar servidor + túnel { display-mode: "form" }
#@markdown Deja esta celda corriendo. A las `MAX_HORAS` se apaga sola, guarda en Drive y libera la sesión.

import json, time, threading, subprocess, re, sys, tarfile, urllib.request, urllib.error
from pathlib import Path

SERVER_PROC = None
TUNNEL_PROC = None
STOP = threading.Event()
READY = threading.Event()
SEND_LOCK = threading.Lock()
session_start = time.time()
ANSI = re.compile(r"\x1b\[[0-9;]*m")


def stream(proc, prefix="", on_line=None):
    try:
        for line in iter(proc.stdout.readline, ""):
            if not line:
                break
            clean = ANSI.sub("", line.rstrip())
            print(prefix + clean)
            if on_line:
                on_line(clean)
            if "Done (" in clean:
                READY.set()
    except Exception:
        pass


def send(cmd):
    with SEND_LOCK:
        if SERVER_PROC and SERVER_PROC.poll() is None:
            try:
                SERVER_PROC.stdin.write(cmd + "\n")
                SERVER_PROC.stdin.flush()
                print(f"⌨️  > {cmd}")
            except Exception as exc:
                print(f"(no pude enviar {cmd!r}: {exc})")


def _bore():
    """Túnel público sin cuenta: bore.pub:PUERTO (la dirección cambia cada sesión)."""
    global TUNNEL_PROC
    import platform
    machine = platform.machine().lower()
    arch = "aarch64" if machine in ("arm64", "aarch64") else "x86_64"
    target = "apple-darwin" if sys.platform == "darwin" else "unknown-linux-musl"
    url = (f"https://github.com/ekzhang/bore/releases/download/v0.6.0/"
           f"bore-v0.6.0-{arch}-{target}.tar.gz")
    exe = Path("/content/bore") if IN_COLAB else Path("/private/tmp/opencode/bore")
    if not exe.exists():
        tgz = Path("/tmp/bore.tgz")
        asset = url.rsplit("/", 1)[-1]
        download(url, tgz, sha256=github_sha256("ekzhang/bore", "v0.6.0", asset))
        safe_extract(tgz, exe.parent)
        built = exe.parent / "bore"
        if built != exe and built.exists():
            built.rename(exe)
        exe.chmod(0o755)

    shown = set()

    def watch(line):
        match = re.search(r"listening at (bore\.pub:\d+)", line)
        if match and "addr" not in shown:
            shown.add("addr")
            print("\n" + "=" * 62)
            print("🔗 DIRECCIÓN PARA TUS AMIGOS:  " + match.group(1))
            print("=" * 62 + "\n")

    TUNNEL_PROC = subprocess.Popen(
        [str(exe), "local", "25565", "--to", "bore.pub"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    threading.Thread(target=stream, args=(TUNNEL_PROC, "[bore] "),
                     kwargs={"on_line": watch}, daemon=True).start()
    print("🔗 bore.pub: túnel temporal, sin cuenta ni configuración.")


PLAYIT_VERSION = "v0.17.1"
PLAYIT_SHA256 = "e78d463d93aa1e3ec36a06ded5a1f4fe879905fdceb865df8f4cef6124f8a555"


def _playit_secret(path):
    """Lee la clave del agente desde el playit.toml guardado en Drive."""
    try:
        text = Path(path).read_text()
    except Exception:
        return None
    match = re.search(r'secret_key\s*=\s*"([0-9a-fA-F]+)"', text)
    if match:
        return match.group(1)
    text = text.strip()
    return text if re.fullmatch(r"[0-9a-fA-F]{16,}", text) else None


def _playit_address(secret):
    """Pregunta a la API de playit la dirección pública de tus túneles."""
    body = b"{}"
    warned = False
    hinted = False
    for _ in range(60):
        if STOP.wait(5):
            return
        try:
            req = urllib.request.Request(
                "https://api.playit.gg/v1/agents/rundata",
                data=body,
                headers={
                    "Authorization": "Agent-Key " + secret,
                    "Content-Type": "application/json",
                    "User-Agent": "MineColab/1.0",
                },
                method="POST",
            )
            with urllib.request.urlopen(req, timeout=15) as r:
                reply = json.load(r)
        except urllib.error.HTTPError as exc:
            try:
                reply = json.loads(exc.read().decode("utf-8", "replace"))
            except Exception:
                reply = {"status": "error", "data": {}}
        except Exception:
            continue
        if reply.get("status") != "success":
            if not warned:
                warned = True
                detail = (reply.get("data") or {}).get("message", "clave no válida")
                print(f"   (playit todavía no acepta la clave: {detail}; reintentando…)")
            continue
        payload = reply.get("data") or {}
        tunnels = [t for t in payload.get("tunnels") or [] if t.get("display_address")]
        if tunnels:
            print("\n" + "=" * 62)
            for t in tunnels:
                label = t.get("name") or t.get("tunnel_type_display") or "túnel"
                print(f"🔗 DIRECCIÓN DE PLAYIT ({label}): {t['display_address']}")
            print("=" * 62)
            print("   Si no ves el túnel de Minecraft, créalo en playit.gg:"
                  " 'Minecraft Java' → 127.0.0.1:25565\n")
            return
        if not hinted:
            hinted = True
            print("\nℹ️  Tu agente de playit está conectado pero aún no tiene túneles.")
            print("   Entra en playit.gg → tu agente → crea un túnel 'Minecraft Java'")
            print("   apuntando a 127.0.0.1:25565. La dirección saldrá aquí en ~10 s.\n")
        if payload.get("pending"):
            print("⏳ playit está creando el túnel…")


def _playit():
    """playit.gg: dirección fija. La clave del agente vive en tu Drive."""
    global TUNNEL_PROC
    cfg = DRIVE_ROOT / "playit"
    cfg.mkdir(parents=True, exist_ok=True)
    secret_file = cfg / "playit.toml"

    # Migración: si ya tenías la config en la sesión anterior dentro de ~/.config, la traemos.
    old = Path.home() / ".config" / "playit_gg" / "playit.toml"
    if not secret_file.exists() and old.exists():
        shutil.copy(old, secret_file)

    # OJO: desde v1.0 el binario 'playit-linux-amd64' es solo el daemon y no imprime
    # el enlace de claim. Usamos v0.17.1, que sigue soportada y funciona sola.
    exe = (Path("/content") if IN_COLAB else Path("/private/tmp/opencode")) / "playit-0.17.1"
    if not exe.exists():
        url = (f"https://github.com/playit-cloud/playit-agent/releases/download/"
               f"{PLAYIT_VERSION}/playit-linux-amd64")
        download(url, exe, sha256=github_sha256("playit-cloud/playit-agent",
                                                PLAYIT_VERSION, "playit-linux-amd64")
                                  or PLAYIT_SHA256)
        exe.chmod(0o755)

    state = {"claim": False, "tunnel": False}

    def watch(line):
        match = re.search(r"https://playit\.gg/claim/[0-9a-fA-F]+", line)
        if match and not state["claim"]:
            state["claim"] = True
            print("\n" + "=" * 62)
            print("🔗 VINCULA EL AGENTE (solo la 1ª vez): " + match.group(0))
            print("   Ábrelo, aprueba el agente y espera; luego crea un túnel")
            print("   'Minecraft Java' → 127.0.0.1:25565 en playit.gg")
            print("=" * 62 + "\n")
        if "tunnel running" in line and not state["tunnel"]:
            state["tunnel"] = True
            print("✅ playit conectado. Buscando tu dirección pública…")
            key = _playit_secret(secret_file)
            if key:
                threading.Thread(target=_playit_address, args=(key,), daemon=True).start()

    # -s = logs planos (sin la interfaz de pantalla completa, que en Colab no se ve).
    # --secret_path = la clave se guarda en Drive y sobrevive entre sesiones.
    TUNNEL_PROC = subprocess.Popen(
        [str(exe), "-s", "--secret_path", str(secret_file)],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    threading.Thread(target=stream, args=(TUNNEL_PROC, "[playit] "),
                     kwargs={"on_line": watch}, daemon=True).start()
    print("🔗 playit.gg arrancando…")
    print("   · 1ª vez: abre el enlace 'claim' de abajo y aprueba el agente.")
    print("   · Luego crea un túnel 'Minecraft Java' → 127.0.0.1:25565.")
    print("   · La clave del agente se guarda en tu Drive para la próxima sesión.\n")

    def supervise():
        waited = 0
        while waited < 120 and not STOP.wait(5):
            waited += 5
            if TUNNEL_PROC.poll() is not None:
                break
            if state["claim"] or state["tunnel"]:
                return  # hay progreso: esperamos a que el usuario apruebe
        if TUNNEL_PROC.poll() is not None or not state["tunnel"]:
            print("\n⚠️  playit no llegó a conectar. Paso automáticamente a bore…")
            try:
                _bore()
            except Exception as exc:
                print("   tampoco pude abrir bore:", exc)

    threading.Thread(target=supervise, daemon=True).start()


def start_tunnel():
    if TUNNEL == "ninguno":
        print("🔌 Sin túnel: solo accesible dentro de la sesión.")
        return
    try:
        if TUNNEL == "bore":
            _bore()
        elif TUNNEL == "ngrok":
            sh("pip -q install pyngrok", check=True)
            from pyngrok import conf, ngrok
            if not NGROK_AUTHTOKEN:
                raise RuntimeError("Falta NGROK_AUTHTOKEN en la celda 1b.")
            conf.get_default().auth_token = NGROK_AUTHTOKEN
            tunnel = ngrok.connect(25565, "tcp")
            print("\n🔗 DIRECCIÓN PARA TUS AMIGOS: " + str(tunnel.public_url).replace("tcp://", "") + "\n")
        else:
            _playit()
    except Exception as exc:
        print(f"\n⚠️  El túnel '{TUNNEL}' falló: {exc}")
        if TUNNEL != "bore":
            print("   Probando bore como alternativa…")
            _bore()


def backup_loop():
    while not STOP.wait(BACKUP_MIN * 60):
        send("save-off")
        send("save-all flush")
        time.sleep(3)
        try:
            sync_worlds()
            print("   💾 mundo sincronizado con Drive")
        finally:
            send("save-on")


def mirror_startup_log():
    """Copia el log a Drive mientras arranca, para saber qué pasó si Colab se cae."""
    origen = SERVER_DIR / "logs" / "latest.log"
    destino = DRIVE_ROOT / "arranque.log"
    while not READY.is_set() and not STOP.wait(15):
        try:
            if origen.exists():
                shutil.copy(origen, destino)
        except Exception:
            pass


def reportar_fallo():
    """Si el servidor murió solo, muestra y guarda las últimas líneas del log."""
    code = SERVER_PROC.returncode if SERVER_PROC else None
    if code in (None, 0) or STOP.is_set():
        return
    origen = SERVER_DIR / "logs" / "latest.log"
    if not origen.exists():
        return
    print(f"\n❌ El servidor terminó con código {code}. Últimas líneas de su log:\n")
    lineas = origen.read_text(errors="replace").splitlines()[-40:]
    print("\n".join(lineas))
    if "OutOfMemoryError" in "\n".join(lineas):
        print("\n🧠 Es falta de RAM: baja RAM_GB o quita mods pesados.")
    try:
        shutil.copy(origen, DRIVE_ROOT / "ultimo-error.log")
        print(f"\n📄 Log completo guardado en {DRIVE_ROOT / 'ultimo-error.log'}")
    except Exception:
        pass


def watchdog():
    deadline = time.time() + float(MAX_HORAS) * 3600
    while not STOP.wait(20):
        if time.time() >= deadline:
            send("say El servidor se apagará en 60 s (límite de sesión de Colab).")
            print(f"\n⏰ {MAX_HORAS} h cumplidas: apagando y guardando…")
            time.sleep(60)
            send("stop")
            return


try:
    if not (SERVER_DIR / "eula.txt").exists():
        raise RuntimeError("Falta la instalación. Ejecuta antes las celdas 2 y 3.")

    argv = launch_argv()
    print("🚀 " + " ".join(argv))
    SERVER_PROC = subprocess.Popen(
        argv, cwd=str(SERVER_DIR), stdin=subprocess.PIPE, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    threading.Thread(target=stream, args=(SERVER_PROC, ""), daemon=True).start()
    threading.Thread(target=mirror_startup_log, daemon=True).start()

    mark = time.time()
    while not READY.wait(1):
        if SERVER_PROC.poll() is not None:
            print("\n❌ El servidor terminó antes de arrancar. Mira el log de arriba.")
            break
        if time.time() - mark > 600:
            print("⏳ Sigue arrancando… (la 1ª vez se descargan librerías)")
            mark = time.time()

    if SERVER_PROC.poll() is None:
        start_tunnel()
        time.sleep(3)
        for name in WHITELIST:
            if re.fullmatch(r"[A-Za-z0-9_]{1,16}", str(name)):
                send(f"whitelist add {name}")
        for name in OPS:
            if re.fullmatch(r"[A-Za-z0-9_]{1,16}", str(name)):
                send(f"op {name}")

        print("\n" + "─" * 60)
        print(f"✅ Servidor arriba (Java {JAVA_MAJOR}, {globals().get('RAM_EFECTIVA_GB', RAM_GB)} GB, {SERVER_TYPE} {MC_VERSION})")
        print(f"   Se apagará solo a las {MAX_HORAS} h o cuando pares esta celda.")
        print("   Comparte la dirección del túnel con tus amigos y deja la pestaña abierta.")
        if TUNNEL != "ninguno" and not WHITELIST:
            print("   ⚠️  Sin whitelist: cualquiera con la dirección puede entrar.")
            print("      Añade nombres en WHITELIST (celda 1b) para cerrarlo.")
        if not ONLINE_MODE:
            print("   ⚠️  online-mode=false: cualquiera puede usar el nombre de otro jugador.")
        print("─" * 60 + "\n")

        threading.Thread(target=backup_loop, daemon=True).start()
        threading.Thread(target=watchdog, daemon=True).start()

        while SERVER_PROC.poll() is None:
            time.sleep(1)
        print(f"\nℹ️  El servidor terminó con código {SERVER_PROC.returncode}.")
except KeyboardInterrupt:
    print("\n⌨️  Interrumpido desde el notebook.")
finally:
    reportar_fallo()
    STOP.set()
    if SERVER_PROC and SERVER_PROC.poll() is None:
        print("\n🛑 Deteniendo el servidor (guardando mundo)…")
        send("stop")
        try:
            SERVER_PROC.wait(timeout=120)
        except subprocess.TimeoutExpired:
            print("   no cerró a tiempo, forzando")
            SERVER_PROC.kill()
    if TUNNEL_PROC:
        TUNNEL_PROC.terminate()
        try:
            TUNNEL_PROC.wait(timeout=10)
        except subprocess.TimeoutExpired:
            TUNNEL_PROC.kill()
    sync_to_drive()
    backup_world("fin")
    print(f"\n✅ Sesión cerrada ({(time.time() - session_start) / 60:.0f} min). Mundo guardado en Drive.")


In [ ]:
#@title 🔎 7. Buscar mods compatibles en Modrinth { display-mode: "form" }
#@markdown Busca por nombre y copia el *slug* que te guste en `MODS` (celda 1b). No instala nada.

import json, urllib.parse

MODRINTH = globals().get("MODRINTH", "https://api.modrinth.com/v2")
QUERY = "create" #@param {type:"string"}
CANTIDAD = 10 #@param {type:"slider", min:5, max:25, step:5}

facets = [["project_type:mod"], [f"versions:{MC_VERSION}"]]
loader_tag = {"fabric": "fabric", "forge": "forge", "neoforge": "neoforge", "paper": "paper"}.get(SERVER_TYPE)
if loader_tag:
    facets.append([f"categories:{loader_tag}"])

query = urllib.parse.urlencode({"query": QUERY, "limit": CANTIDAD, "facets": json.dumps(facets)})
results = get_json(f"{MODRINTH}/search?{query}")["hits"]

print(f"Resultados para '{QUERY}' en {SERVER_TYPE} {MC_VERSION}:\n")
for hit in results:
    server_side = {"required": "servidor", "optional": "ambos", "unsupported": "solo cliente"}.get(
        hit.get("server_side"), "?"
    )
    print(f"  {hit['slug']:<28} {hit['title'][:34]:<34} {hit['downloads']:>10,} descargas · {server_side}")

print("\nCopia los slugs en MODS (celda 1b) y vuelve a ejecutar la celda 4.")


In [ ]:
#@title 🧰 8. Extras: guardar, backups, restaurar, ver log { display-mode: "form" }
#@markdown Con el servidor parado. `ACCION` hace lo que dice; `BACKUP` es el nombre exacto del archivo.

from pathlib import Path
import re

ACCION = "solo listar" #@param ["solo listar", "guardar ahora", "backup del mundo", "restaurar backup", "ver últimas líneas del log"]
BACKUP = "" #@param {type:"string"}

running = globals().get("SERVER_PROC")
if running is not None and running.poll() is None:
    print("⚠️  El servidor sigue corriendo. Para esta celda primero (o usa /stop).")

if ACCION == "guardar ahora":
    sync_to_drive()
elif ACCION == "backup del mundo":
    backup_world("manual")
    sync_to_drive()
elif ACCION == "restaurar backup":
    if not BACKUP:
        print("Escribe el nombre del backup en el campo BACKUP (mira la lista de abajo).")
    elif not re.fullmatch(r"world-[A-Za-z0-9_.-]+\.tar\.gz", BACKUP):
        print("Nombre de backup no válido.")
    else:
        origin = BACKUP_DIR / BACKUP
        if not origin.exists():
            print("No existe:", origin)
        else:
            trash = SERVER_DIR.parent / ("mundos-viejos-" + datetime.now().strftime("%Y%m%d-%H%M%S"))
            trash.mkdir(parents=True, exist_ok=True)
            for world in SERVER_DIR.glob("world*"):
                if world.is_dir():
                    shutil.move(str(world), str(trash / world.name))
            safe_extract(origin, SERVER_DIR)
            print(f"✅ Restaurado {origin.name} (lo anterior quedó en {trash})")
            sync_to_drive()
elif ACCION == "ver últimas líneas del log":
    log = SERVER_DIR / "logs" / "latest.log"
    print("\n".join(log.read_text(errors="ignore").splitlines()[-60:]) if log.exists() else "Todavía no hay logs.")

backups = sorted(BACKUP_DIR.glob("world-*.tar.gz"))
print("\n💾 Backups en Drive (se conservan los 3 últimos):")
for b in backups:
    print(f"   {b.name:<42} {b.stat().st_size / 1e6:>7.0f} MB")
print(f"\n📁 local: {SERVER_DIR}\n☁️  Drive: {DRIVE_ROOT}")
